## STEP 1 — Mount Drive



In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## STEP 2 — Imports and Project Paths

In [ ]:
from pathlib import Path
import os
import zipfile
import json
import shutil
from collections import Counter

In [ ]:
# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Image_captioning"
)

DATASET_DIR = PROJECT_ROOT / "Dataset"

ZIP_DIR = DATASET_DIR / "ZIP"
RAW_DIR = DATASET_DIR / "Raw"
PROCESSED_DIR = DATASET_DIR / "Processed"

NOTEBOOK_DIR = PROJECT_ROOT / "Notebooks"
FEATURES_DIR = PROJECT_ROOT / "Features"
MODELS_DIR = PROJECT_ROOT / "Models"

In [ ]:
# ============================================================
# CREATE ONLY REQUIRED DIRECTORIES
# ============================================================

for directory in [
    PROJECT_ROOT,
    DATASET_DIR,
    ZIP_DIR,
    RAW_DIR,
    PROCESSED_DIR,
    NOTEBOOK_DIR,
    FEATURES_DIR,
    MODELS_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

In [ ]:
print("=" * 70)
print("IMAGE CAPTIONING PROJECT — NOTEBOOK 1")
print("=" * 70)

print("\nProject root:")
print(PROJECT_ROOT)

print("\nZIP directory:")
print(ZIP_DIR)

print("\nRaw directory:")
print(RAW_DIR)

print("\n✓ Project paths ready")

IMAGE CAPTIONING PROJECT — NOTEBOOK 1

Project root:
/content/drive/MyDrive/Image_captioning

ZIP directory:
/content/drive/MyDrive/Image_captioning/Dataset/ZIP

Raw directory:
/content/drive/MyDrive/Image_captioning/Dataset/Raw

✓ Project paths ready


## STEP 3 — Detect ZIP Files

In [ ]:
# ============================================================
# FIND ALL ZIP FILES
# ============================================================

zip_files = sorted(
    ZIP_DIR.glob("*.zip")
)

print("=" * 70)
print("ZIP FILE DETECTION")
print("=" * 70)

print(f"\nZIP directory: {ZIP_DIR}")
print(f"ZIP files found: {len(zip_files)}\n")

if len(zip_files) == 0:
    print("❌ NO ZIP FILES FOUND")

else:
    for i, file in enumerate(zip_files, start=1):

        size_gb = (
            file.stat().st_size
            / (1024 ** 3)
        )

        size_mb = (
            file.stat().st_size
            / (1024 ** 2)
        )

        if size_gb >= 1:
            size_text = f"{size_gb:.2f} GB"
        else:
            size_text = f"{size_mb:.2f} MB"

        print(
            f"{i}. {file.name}"
        )

        print(
            f"   Size: {size_text}"
        )


assert len(zip_files) >= 2, (
    "Expected at least two ZIP files "
    "(Flickr8K and Flickr30K)."
)

print("\n✓ ZIP files detected")

ZIP FILE DETECTION

ZIP directory: /content/drive/MyDrive/Image_captioning/Dataset/ZIP
ZIP files found: 2

1. flickr30k.zip
   Size: 8.16 GB
2. flickr8k.zip
   Size: 1.04 GB

✓ ZIP files detected


## STEP 4 — Inspect ZIP Contents Before Extraction

In [ ]:
# ============================================================
# INSPECT ZIP CONTENTS
# ============================================================

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png"
}

print("=" * 70)
print("ZIP CONTENT INSPECTION")
print("=" * 70)


zip_information = {}


for zip_path in zip_files:

    print("\n" + "=" * 70)
    print(f"ZIP FILE: {zip_path.name}")
    print("=" * 70)

    with zipfile.ZipFile(
        zip_path,
        "r"
    ) as archive:

        members = archive.namelist()

        image_members = [
            member
            for member in members
            if Path(member).suffix.lower()
            in IMAGE_EXTENSIONS
        ]

        file_members = [
            member
            for member in members
            if not member.endswith("/")
        ]


        zip_information[
            zip_path.name
        ] = {
            "total_members": len(members),
            "total_files": len(file_members),
            "image_files": len(image_members),
            "sample_members": members[:20]
        }


        print(
            f"\nTotal archive members: {len(members)}"
        )

        print(
            f"Total files: {len(file_members)}"
        )

        print(
            f"Image files: {len(image_members)}"
        )

        print("\nFirst 20 archive paths:\n")

        for member in members[:20]:
            print(member)


print("\n" + "=" * 70)
print("ZIP INSPECTION COMPLETE")
print("=" * 70)

ZIP CONTENT INSPECTION

ZIP FILE: flickr30k.zip

Total archive members: 63568
Total files: 63568
Image files: 63566

First 20 archive paths:

flickr30k_images/flickr30k_images/1000092795.jpg
flickr30k_images/flickr30k_images/10002456.jpg
flickr30k_images/flickr30k_images/1000268201.jpg
flickr30k_images/flickr30k_images/1000344755.jpg
flickr30k_images/flickr30k_images/1000366164.jpg
flickr30k_images/flickr30k_images/1000523639.jpg
flickr30k_images/flickr30k_images/1000919630.jpg
flickr30k_images/flickr30k_images/10010052.jpg
flickr30k_images/flickr30k_images/1001465944.jpg
flickr30k_images/flickr30k_images/1001545525.jpg
flickr30k_images/flickr30k_images/1001573224.jpg
flickr30k_images/flickr30k_images/1001633352.jpg
flickr30k_images/flickr30k_images/1001773457.jpg
flickr30k_images/flickr30k_images/1001896054.jpg
flickr30k_images/flickr30k_images/100197432.jpg
flickr30k_images/flickr30k_images/100207720.jpg
flickr30k_images/flickr30k_images/1002674143.jpg
flickr30k_images/flickr30k_imag

## STEP 5 — Verify ZIP Integrity

In [ ]:
# ============================================================
# ZIP INTEGRITY CHECK
# ============================================================

print("=" * 70)
print("ZIP INTEGRITY CHECK")
print("=" * 70)


for zip_path in zip_files:

    print(
        f"\nChecking: {zip_path.name}"
    )

    with zipfile.ZipFile(
        zip_path,
        "r"
    ) as archive:

        bad_file = archive.testzip()


    if bad_file is None:

        print(
            "✓ ZIP integrity verified"
        )

    else:

        raise RuntimeError(
            f"❌ Corrupted file found inside "
            f"{zip_path.name}: {bad_file}"
        )


print("\n" + "=" * 70)
print("✓ ALL ZIP FILES PASSED INTEGRITY CHECK")
print("=" * 70)

ZIP INTEGRITY CHECK

Checking: flickr30k.zip
✓ ZIP integrity verified

Checking: flickr8k.zip
✓ ZIP integrity verified

✓ ALL ZIP FILES PASSED INTEGRITY CHECK


##STEP 6 — Identify Flickr8K and Flickr30K Automatically

In [ ]:
# ============================================================
# IDENTIFY DATASETS
# ============================================================

def identify_dataset(
    filename
):
    """
    Identify Flickr8K or Flickr30K from filename.
    """

    name = filename.lower()

    if (
        "30k" in name
        or "flickr30" in name
        or "flickr_30" in name
    ):
        return "Flickr30K"

    elif (
        "8k" in name
        or "flickr8" in name
        or "flickr_8" in name
    ):
        return "Flickr8K"

    else:
        return None


dataset_zip_map = {}


print("=" * 70)
print("DATASET IDENTIFICATION")
print("=" * 70)


for zip_path in zip_files:

    dataset_name = identify_dataset(
        zip_path.name
    )

    print(
        f"\nFile: {zip_path.name}"
    )

    print(
        f"Detected dataset: {dataset_name}"
    )


    if dataset_name is not None:

        if dataset_name in dataset_zip_map:

            raise ValueError(
                f"Multiple ZIP files detected "
                f"for {dataset_name}."
            )

        dataset_zip_map[
            dataset_name
        ] = zip_path


expected_datasets = {
    "Flickr8K",
    "Flickr30K"
}


missing_datasets = (
    expected_datasets
    - set(dataset_zip_map.keys())
)


if len(missing_datasets) > 0:

    print(
        "\n⚠️ Automatic identification failed "
        "for:"
    )

    for dataset in missing_datasets:
        print(f"• {dataset}")

    print(
        "\nSTOP HERE and send me the output."
    )

else:

    print("\n" + "=" * 70)
    print("✓ BOTH DATASETS IDENTIFIED")
    print("=" * 70)

    for dataset_name, zip_path in (
        dataset_zip_map.items()
    ):

        print(
            f"{dataset_name}: "
            f"{zip_path.name}"
        )

DATASET IDENTIFICATION

File: flickr30k.zip
Detected dataset: Flickr30K

File: flickr8k.zip
Detected dataset: Flickr8K

✓ BOTH DATASETS IDENTIFIED
Flickr30K: flickr30k.zip
Flickr8K: flickr8k.zip


##STEP 7 — Define Extraction Locations

In [ ]:
# ============================================================
# EXTRACTION PATHS
# ============================================================

FLICKR8K_RAW_DIR = (
    RAW_DIR / "Flickr8K"
)

FLICKR30K_RAW_DIR = (
    RAW_DIR / "Flickr30K"
)


FLICKR8K_RAW_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FLICKR30K_RAW_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("=" * 70)
print("EXTRACTION PATHS")
print("=" * 70)

print(
    "\nFlickr8K extraction directory:"
)

print(
    FLICKR8K_RAW_DIR
)

print(
    "\nFlickr30K extraction directory:"
)

print(
    FLICKR30K_RAW_DIR
)


print(
    "\n✓ Extraction directories ready"
)

EXTRACTION PATHS

Flickr8K extraction directory:
/content/drive/MyDrive/Image_captioning/Dataset/Raw/Flickr8K

Flickr30K extraction directory:
/content/drive/MyDrive/Image_captioning/Dataset/Raw/Flickr30K

✓ Extraction directories ready


## STEP 8 — Check Whether Data Is Already Extracted

In [ ]:
# ============================================================
# CHECK EXISTING EXTRACTION
# ============================================================

def count_images(
    directory
):
    """
    Count supported image files recursively.
    """

    return sum(
        1
        for file in directory.rglob("*")
        if (
            file.is_file()
            and file.suffix.lower()
            in IMAGE_EXTENSIONS
        )
    )


print("=" * 70)
print("EXISTING EXTRACTION CHECK")
print("=" * 70)


flickr8k_existing_images = (
    count_images(
        FLICKR8K_RAW_DIR
    )
)

flickr30k_existing_images = (
    count_images(
        FLICKR30K_RAW_DIR
    )
)


print(
    f"\nFlickr8K existing images : "
    f"{flickr8k_existing_images:,}"
)

print(
    f"Flickr30K existing images: "
    f"{flickr30k_existing_images:,}"
)


print("\nDecision:")


if flickr8k_existing_images > 0:

    print(
        "✓ Flickr8K appears extracted"
    )

else:

    print(
        "→ Flickr8K needs extraction"
    )


if flickr30k_existing_images > 0:

    print(
        "✓ Flickr30K appears extracted"
    )

else:

    print(
        "→ Flickr30K needs extraction"
    )

EXISTING EXTRACTION CHECK

Flickr8K existing images : 0
Flickr30K existing images: 0

Decision:
→ Flickr8K needs extraction
→ Flickr30K needs extraction


## STEP 9 — Extract Flickr8K

In [ ]:
# ============================================================
# EXTRACT FLICKR8K
# ============================================================

if flickr8k_existing_images > 0:

    print("=" * 70)
    print("FLICKR8K ALREADY EXTRACTED")
    print("=" * 70)

    print(
        f"Images already found: "
        f"{flickr8k_existing_images:,}"
    )


else:

    print("=" * 70)
    print("EXTRACTING FLICKR8K")
    print("=" * 70)

    flickr8k_zip = (
        dataset_zip_map[
            "Flickr8K"
        ]
    )


    with zipfile.ZipFile(
        flickr8k_zip,
        "r"
    ) as archive:

        members = archive.infolist()

        total_members = len(
            members
        )


        for i, member in enumerate(
            members,
            start=1
        ):

            archive.extract(
                member,
                FLICKR8K_RAW_DIR
            )


            if (
                i % 1000 == 0
                or i == total_members
            ):

                print(
                    f"Extracted "
                    f"{i:,}/{total_members:,} files"
                )


    print(
        "\n✓ Flickr8K extraction complete"
    )

EXTRACTING FLICKR8K
Extracted 1,000/8,092 files
Extracted 2,000/8,092 files
Extracted 3,000/8,092 files
Extracted 4,000/8,092 files
Extracted 5,000/8,092 files
Extracted 6,000/8,092 files
Extracted 7,000/8,092 files
Extracted 8,000/8,092 files
Extracted 8,092/8,092 files

✓ Flickr8K extraction complete


##STEP 10 — Verify Flickr8K Extraction

In [ ]:
# ============================================================
# VERIFY FLICKR8K EXTRACTION
# ============================================================

print("=" * 70)
print("VERIFYING FLICKR8K")
print("=" * 70)


flickr8k_images = sorted(
    file
    for file in FLICKR8K_RAW_DIR.rglob("*")
    if (
        file.is_file()
        and file.suffix.lower()
        in IMAGE_EXTENSIONS
    )
)


print(
    f"\nTotal images found: "
    f"{len(flickr8k_images):,}"
)


if len(flickr8k_images) > 0:

    print(
        "\nFirst 5 images:"
    )

    for image_path in (
        flickr8k_images[:5]
    ):

        print(
            image_path.relative_to(
                FLICKR8K_RAW_DIR
            )
        )


else:

    raise RuntimeError(
        "❌ No images found after "
        "Flickr8K extraction."
    )


print(
    "\n✓ Flickr8K images verified"
)

VERIFYING FLICKR8K

Total images found: 8,091

First 5 images:
Images/1000268201_693b08cb0e.jpg
Images/1001773457_577c3a7d70.jpg
Images/1002674143_1b742ab4b8.jpg
Images/1003163366_44323f5815.jpg
Images/1007129816_e794419615.jpg

✓ Flickr8K images verified


## STEP 11 — Extract Flickr30K

In [ ]:
# ============================================================
# EXTRACT FLICKR30K
# ============================================================

if flickr30k_existing_images > 0:

    print("=" * 70)
    print("FLICKR30K ALREADY EXTRACTED")
    print("=" * 70)

    print(
        f"Images already found: "
        f"{flickr30k_existing_images:,}"
    )


else:

    print("=" * 70)
    print("EXTRACTING FLICKR30K")
    print("=" * 70)

    flickr30k_zip = (
        dataset_zip_map[
            "Flickr30K"
        ]
    )


    with zipfile.ZipFile(
        flickr30k_zip,
        "r"
    ) as archive:

        members = archive.infolist()

        total_members = len(
            members
        )


        for i, member in enumerate(
            members,
            start=1
        ):

            archive.extract(
                member,
                FLICKR30K_RAW_DIR
            )


            if (
                i % 5000 == 0
                or i == total_members
            ):

                print(
                    f"Extracted "
                    f"{i:,}/{total_members:,} files"
                )


    print(
        "\n✓ Flickr30K extraction complete"
    )

EXTRACTING FLICKR30K
Extracted 5,000/63,568 files
Extracted 10,000/63,568 files
Extracted 15,000/63,568 files
Extracted 20,000/63,568 files
Extracted 25,000/63,568 files


## STEP 12 — Verify Flickr30K Extraction

In [ ]:
# ============================================================
# VERIFY FLICKR30K EXTRACTION
# ============================================================

print("=" * 70)
print("VERIFYING FLICKR30K")
print("=" * 70)


flickr30k_images = sorted(
    file
    for file in FLICKR30K_RAW_DIR.rglob("*")
    if (
        file.is_file()
        and file.suffix.lower()
        in IMAGE_EXTENSIONS
    )
)


print(
    f"\nTotal images found: "
    f"{len(flickr30k_images):,}"
)


if len(flickr30k_images) > 0:

    print(
        "\nFirst 5 images:"
    )

    for image_path in (
        flickr30k_images[:5]
    ):

        print(
            image_path.relative_to(
                FLICKR30K_RAW_DIR
            )
        )


else:

    raise RuntimeError(
        "❌ No images found after "
        "Flickr30K extraction."
    )


print(
    "\n✓ Flickr30K images verified"
)

##STEP 13 — Find Caption / Annotation Files

In [ ]:
# ============================================================
# FIND POSSIBLE CAPTION / ANNOTATION FILES
# ============================================================

TEXT_EXTENSIONS = {
    ".txt",
    ".csv",
    ".json",
    ".xml"
}


def find_annotation_files(
    dataset_dir
):
    """
    Find possible annotation and caption files.
    """

    files = []

    for file in dataset_dir.rglob("*"):

        if not file.is_file():
            continue


        name = file.name.lower()


        if (
            file.suffix.lower()
            in TEXT_EXTENSIONS
        ):

            if any(
                keyword in name
                for keyword in [
                    "caption",
                    "token",
                    "result",
                    "annotation",
                    "sentence",
                    "dataset",
                    "image"
                ]
            ):

                files.append(
                    file
                )


    return sorted(
        files
    )


print("=" * 70)
print("ANNOTATION FILE SEARCH")
print("=" * 70)


flickr8k_annotations = (
    find_annotation_files(
        FLICKR8K_RAW_DIR
    )
)

flickr30k_annotations = (
    find_annotation_files(
        FLICKR30K_RAW_DIR
    )
)


print(
    "\nFlickr8K possible annotation files:"
)

if len(
    flickr8k_annotations
) == 0:

    print("None found")

else:

    for file in (
        flickr8k_annotations
    ):

        print(
            file.relative_to(
                FLICKR8K_RAW_DIR
            )
        )


print(
    "\nFlickr30K possible annotation files:"
)

if len(
    flickr30k_annotations
) == 0:

    print("None found")

else:

    for file in (
        flickr30k_annotations
    ):

        print(
            file.relative_to(
                FLICKR30K_RAW_DIR
            )
        )

## STEP 14 — Check Image File Integrity (Sample)

In [ ]:
# ============================================================
# SAMPLE IMAGE READABILITY TEST
# ============================================================

from PIL import Image
import random


def verify_image_sample(
    image_paths,
    sample_size=100
):
    """
    Verify that a random sample
    of images can be opened.
    """

    if len(image_paths) == 0:

        raise ValueError(
            "No images available "
            "for verification."
        )


    sample_size = min(
        sample_size,
        len(image_paths)
    )


    sample_paths = random.sample(
        image_paths,
        sample_size
    )


    corrupted = []


    for image_path in sample_paths:

        try:

            with Image.open(
                image_path
            ) as image:

                image.verify()


        except Exception as error:

            corrupted.append(
                (
                    str(image_path),
                    str(error)
                )
            )


    return corrupted


print("=" * 70)
print("IMAGE SAMPLE INTEGRITY TEST")
print("=" * 70)


flickr8k_corrupted = (
    verify_image_sample(
        flickr8k_images,
        sample_size=100
    )
)


flickr30k_corrupted = (
    verify_image_sample(
        flickr30k_images,
        sample_size=100
    )
)


print(
    f"\nFlickr8K sample tested : "
    f"{min(100, len(flickr8k_images))}"
)

print(
    f"Flickr8K failures      : "
    f"{len(flickr8k_corrupted)}"
)


print(
    f"\nFlickr30K sample tested: "
    f"{min(100, len(flickr30k_images))}"
)

print(
    f"Flickr30K failures      : "
    f"{len(flickr30k_corrupted)}"
)


assert len(
    flickr8k_corrupted
) == 0, (
    "Corrupted Flickr8K images "
    "found in sample."
)


assert len(
    flickr30k_corrupted
) == 0, (
    "Corrupted Flickr30K images "
    "found in sample."
)


print(
    "\n✓ Sample image integrity verified"
)

#STEP 15 — Final Notebook 1 Verification Summary

In [ ]:
# ============================================================
# NOTEBOOK 1 — FINAL VERIFICATION
# ============================================================

print("\n")
print("=" * 70)
print("NOTEBOOK 1 — DATASET EXTRACTION COMPLETE")
print("=" * 70)


print("\nPROJECT")

print(
    f"Project root: {PROJECT_ROOT}"
)


print("\nFLICKR8K")

print(
    f"Images: {len(flickr8k_images):,}"
)

print(
    f"Annotation candidates: "
    f"{len(flickr8k_annotations)}"
)

print(
    f"Raw location: "
    f"{FLICKR8K_RAW_DIR}"
)


print("\nFLICKR30K")

print(
    f"Images: {len(flickr30k_images):,}"
)

print(
    f"Annotation candidates: "
    f"{len(flickr30k_annotations)}"
)

print(
    f"Raw location: "
    f"{FLICKR30K_RAW_DIR}"
)


print("\nDATASET STATUS")

assert len(
    flickr8k_images
) > 0

assert len(
    flickr30k_images
) > 0


print(
    "✓ Flickr8K extracted"
)

print(
    "✓ Flickr8K images verified"
)

print(
    "✓ Flickr30K extracted"
)

print(
    "✓ Flickr30K images verified"
)

print(
    "✓ ZIP files retained"
)

print(
    "✓ Raw datasets remain separate"
)


print("\n" + "=" * 70)
print("NOTEBOOK 1 COMPLETE")
print("=" * 70)


print(
    "\nNEXT NOTEBOOK:"
)

print(
    "02_Merge_Dataset.ipynb"
)

print(
    "\nNext phase will:"
)

print(
    "1. Inspect exact annotation formats"
)

print(
    "2. Standardize Flickr8K records"
)

print(
    "3. Standardize Flickr30K records"
)

print(
    "4. Create collision-safe image IDs"
)

print(
    "5. Create unified dataset manifest"
)

print(
    "6. Keep source dataset information"
)